# eval.ipynb — Évaluation finale
Matrice de confusion, AUC-ROC, Recall, F1-score, visualisation des erreurs FN/FP.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '.')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc, confusion_matrix, classification_report
)

%run dataset.ipynb
%run model.ipynb

CHECKPOINT = '../outputs/checkpoints/best_model.pt'
DATA_DIR   = '../data/chest_xray'
FIG_DIR    = Path('../outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Appareil : {DEVICE.upper()}')

## 1. Charger le checkpoint

In [ ]:
ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
arch = ckpt.get('architecture', 'baseline')
model = get_model(arch).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Architecture : {arch}  |  Epoque : {ckpt.get("epoch")}  |  Val loss : {ckpt.get("val_loss", 0):.4f}')

## 2. Inférence sur le test set

In [ ]:
_, _, test_loader = get_dataloaders(DATA_DIR, batch_size=32, num_workers=0)

all_labels, all_probs = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        logits = model(imgs.to(DEVICE)).squeeze(1)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.numpy())

y_true = np.concatenate(all_labels)
y_prob = np.concatenate(all_probs)
y_pred = (y_prob >= 0.5).astype(int)
print(f'Test set : {len(y_true)} images analysees')

## 3. Métriques

In [ ]:
acc   = accuracy_score(y_true, y_pred)
prec  = precision_score(y_true, y_pred, zero_division=0)
rec   = recall_score(y_true, y_pred, zero_division=0)
f1    = f1_score(y_true, y_pred, zero_division=0)
auc_s = roc_auc_score(y_true, y_prob)
cm    = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=== RESULTATS FINAUX ===')
print(f'Accuracy  : {acc:.4f} ({acc*100:.1f}%)')
print(f'Recall    : {rec:.4f}  <- PRIORITAIRE (FNR={fn/(fn+tp):.4f})')
print(f'Precision : {prec:.4f}')
print(f'F1-score  : {f1:.4f}')
print(f'AUC-ROC   : {auc_s:.4f}')
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')
print('\n' + classification_report(y_true, y_pred, target_names=CLASS_NAMES))

## 4. Matrice de confusion + Courbe ROC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Matrice
im = axes[0].imshow(cm, cmap='Blues')
plt.colorbar(im, ax=axes[0])
axes[0].set_xticks(range(2)); axes[0].set_xticklabels(CLASS_NAMES, rotation=30)
axes[0].set_yticks(range(2)); axes[0].set_yticklabels(CLASS_NAMES)
for i in range(2):
    for j in range(2):
        c = 'white' if cm[i,j] > cm.max()/2 else 'black'
        axes[0].text(j, i, str(cm[i,j]), ha='center', va='center', color=c, fontsize=14)
axes[0].set_title('Matrice de confusion'); axes[0].set_ylabel('Reel'); axes[0].set_xlabel('Predit')

# ROC
fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC={auc(fpr,tpr):.4f}')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('Recall (TPR)')
axes[1].set_title('Courbe ROC'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'results_final.png', dpi=150)
%matplotlib inline
plt.show()
print('Saved -> outputs/figures/results_final.png')

## 5. Analyse des erreurs (FN prioritaires)

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406]); STD = np.array([0.229, 0.224, 0.225])
errors = []
with torch.no_grad():
    for imgs, labels in test_loader:
        probs = torch.sigmoid(model(imgs.to(DEVICE)).squeeze(1)).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        for img, lbl, pred, prob in zip(imgs.numpy(), labels.numpy(), preds, probs):
            if lbl != pred:
                errors.append({'img': np.clip(img.transpose(1,2,0)*STD+MEAN,0,1),
                               'true': CLASS_NAMES[lbl], 'pred': CLASS_NAMES[pred],
                               'prob': float(prob), 'type': 'FN' if lbl==1 else 'FP'})
        if len(errors) >= 8: break

if errors:
    n = min(8, len(errors))
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, err in zip(axes.flatten(), errors[:n]):
        ax.imshow(err['img'])
        c = 'red' if err['type'] == 'FN' else 'orange'
        ax.set_title(f"{err['type']} Vrai:{err['true']}\nPredit:{err['pred']} ({err['prob']:.2f})",
                     color=c, fontsize=8)
        ax.axis('off')
    plt.suptitle('Erreurs : FN (rouge) manquent une pneumonie', fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'error_analysis.png', dpi=150)
    %matplotlib inline
    plt.show()
else:
    print('Aucune erreur detectee !')